# 🏆 Gemma Developer Agent: Clean-Bench (Rich) Rigorous A/B Testing Suite

> ### 📌 **Notebook Version: v8.0-clean-rich (Updated: 2026-11-08)**
> - **タスクセットの正常化 (Textualize/rich)**: 環境不備で 100% 失敗する壊れたタスク (FastAPI 27問) を排除し、コミュニティ監査で健全性 39/48 (約81%) が実証された **クリーンな Rich 実タスク** を採用。
> - **3 大仮説の科学的 A/B 検証**:
>   1. **Variant A (V9.2 Baseline / LB 0.17 実績)**: 思考制約なし・波括弧なし・Call 8 強制編集
>   2. **Variant B (V14 Real-Proof / LB 0.12 実績)**: 思考 2 文制約あり・波括弧 One-Shot あり
>   3. **Variant C (V15 候補 / Loop-Breaker)**: 思考制約なし・波括弧なし・**`edit_file` 失敗時の Python 即時迂回ルール** 導入
> - **波括弧 `{` の ADK テンプレート破壊テスト**: ハーネス側で波括弧がどう処理されるかを直接実機検証。

In [ ]:
# 1. 環境初期化 & パッケージ導入 & Kaggle API 最新状況チェック
import os
import sys
import time
import json
import re
import gc
import subprocess
from pathlib import Path

print("[Setup] 依存パッケージをインストール中 (transformers, bitsandbytes, pytest, rich, kaggle, etc.)...")
!pip install --quiet transformers accelerate bitsandbytes compressed-tensors sentencepiece pytest pytest-asyncio requests gitpython kaggle tabulate rich

import torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    device_name = torch.cuda.get_device_name(0)
    print(f"[Device] ✅ GPU 検出: {device_name} (総 VRAM: {vram:.2f} GB)")
else:
    print("[Device ⚠️ WARNING] GPU が無効です。Colabメニューで『A100 GPU (ハイメモリ)』を選択してください。")

# Kaggle API による Day 18 提出状況チェック
try:
    from google.colab import userdata
    k_user = userdata.get('KAGGLE_USERNAME')
    k_key = userdata.get('KAGGLE_KEY')
    if k_user and k_key:
        os.environ['KAGGLE_USERNAME'] = k_user
        os.environ['KAGGLE_KEY'] = k_key
        res = subprocess.run(['kaggle', 'competitions', 'submissions', '-c', 'gemma-4-developer-agent'], capture_output=True, text=True)
        print("\n[Kaggle Submissions Status]:")
        for line in res.stdout.split('\n')[:5]:
            print("  " + line)
except Exception as e:
    print(f"[Kaggle Notice] Kaggle API チェック スキップ: {e}")


In [ ]:
# 2. サンドボックス環境の準備 (Textualize/rich クリーンリポジトリ)
SANDBOX_DIR = "/content/mini_kaggle_sandbox"
REPO_DIR = os.path.join(SANDBOX_DIR, "rich_repo")
os.makedirs(SANDBOX_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    print("[Sandbox] Textualize/rich 公式リポジトリをクローン中...")
    !git clone --quiet https://github.com/Textualize/rich.git {REPO_DIR}
else:
    print("[Sandbox] 既存の Rich リポジトリを使用します。")

print("[Sandbox] Rich のテスト実行環境をセットアップ中...")
!pip install --quiet typing_extensions pygments pytest pytest-cov
print("[Sandbox] ✅ Rich サンドボックス準備完了。")


In [ ]:
# 3. 本番同等モデルのロード (二重安全ネット付き推論エンジン)
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

hf_token = None
try:
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
except Exception:
    pass

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
model = None
tokenizer = None
MODEL_ID = ""

# A100-80GB: 本番公式 31B-QAT / 40GB: 27B-it 4-bit / T4: 9B-it 4-bit
if vram_gb >= 70.0:
    PRIMARY_ID = "google/gemma-4-31B-it-qat-w4a16-ct"
    print(f"[Model] 🚀 A100-80GB 検出 ({vram_gb:.1f} GB): 本番公式 31B モデル ({PRIMARY_ID}) をロード試行中...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(PRIMARY_ID, token=hf_token)
        model = AutoModelForCausalLM.from_pretrained(PRIMARY_ID, device_map={0: "38GiB", "cpu": "60GiB"}, torch_dtype=torch.bfloat16, token=hf_token)
        MODEL_ID = PRIMARY_ID
        print(f"[Model] 🎉 31B-QAT ロード成功！")
    except Exception as e:
        print(f"[Model Warning] 31B ロード失敗 ({e})。27B-it 4-bit へ自動フォールバックします。")
        model = None

if model is None and vram_gb >= 35.0:
    FALLBACK_27B = "google/gemma-2-27b-it"
    print(f"[Model] 🚀 27B モデル ({FALLBACK_27B}) を 4-bit 直結ロード中...")
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    tokenizer = AutoTokenizer.from_pretrained(FALLBACK_27B, token=hf_token)
    model = AutoModelForCausalLM.from_pretrained(FALLBACK_27B, quantization_config=bnb_config, device_map={"": 0}, torch_dtype=torch.bfloat16, token=hf_token)
    MODEL_ID = FALLBACK_27B

if model is None:
    FALLBACK_9B = "google/gemma-2-9b-it"
    print(f"[Model] 🚀 9B モデル ({FALLBACK_9B}) を 4-bit 直結ロード中...")
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16)
    tokenizer = AutoTokenizer.from_pretrained(FALLBACK_9B, token=hf_token)
    model = AutoModelForCausalLM.from_pretrained(FALLBACK_9B, quantization_config=bnb_config, device_map={"": 0}, torch_dtype=torch.bfloat16, token=hf_token)
    MODEL_ID = FALLBACK_9B

alloc = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
print(f"[Model Ready] 🎯 確定稼働モデル: {MODEL_ID} | VRAM 消費: {alloc:.2f} GB / 空き: {vram_gb - alloc:.1f} GB")

def call_llm(messages, temperature=0.2, max_new_tokens=1024):
    """二重安全ネット付きの推論関数 (TemplateError 完全防止 & 確実な発話生成)"""
    normalized = []
    sys_prefix = ""
    for m in messages:
        role = m["role"]
        if role == "system":
            sys_prefix += m["content"].strip() + "\n\n"
            continue
        r = "assistant" if role in ["assistant", "model"] else "user"
        content = m.get("content", "").strip()
        if not content:
            content = "(empty)"
        if r == "user" and sys_prefix:
            content = sys_prefix + content
            sys_prefix = ""
        if not normalized:
            normalized.append({"role": r, "content": content})
        elif normalized[-1]["role"] == r:
            normalized[-1]["content"] += "\n\n" + content
        else:
            normalized.append({"role": r, "content": content})
            
    if not normalized and sys_prefix:
        normalized.append({"role": "user", "content": sys_prefix})
        
    try:
        prompt_text = tokenizer.apply_chat_template(normalized, tokenize=False, add_generation_prompt=True)
    except Exception:
        prompt_text = ""
        for m in normalized:
            tag = "model" if m["role"] == "assistant" else "user"
            prompt_text += f"<start_of_turn>{tag}\n{m['content']}<end_of_turn>\n"
        prompt_text += "<start_of_turn>model\n"
        
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]
    
    t0 = time.time()
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0.0),
            temperature=max(temperature, 1e-4),
            pad_token_id=tokenizer.eos_token_id
        )
    elapsed = time.time() - t0
    resp = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    return resp, elapsed

print("[Engine] ✅ 推論関数 call_llm 準備完了。")


In [ ]:
# 4. 実環境ツール実行エンジン & 万能マルチ形式パーサー (Robust Multi-Format Engine)
import ast
import re
import json
import os
import subprocess

def execute_tool(tool_name, arguments, repo_dir):
    """リポジトリサンドボックス上で実際のツールを実行する (完全型安全 & スマートパス自動解決)"""
    try:
        def sanitize_path(raw_path):
            if not raw_path: return ""
            p = str(raw_path).strip().replace("'", "").replace('"', '')
            for pfx in ["/workspace/", "workspace/", "/"]:
                while p.startswith(pfx):
                    p = p[len(pfx):]
            target = os.path.join(repo_dir, p)
            if os.path.exists(target):
                return target
            parts = p.split("/")
            if len(parts) >= 2 and parts[0] == parts[1]:
                dedup_p = "/".join(parts[1:])
                target_dedup = os.path.join(repo_dir, dedup_p)
                if os.path.exists(target_dedup):
                    return target_dedup
            base = os.path.basename(p)
            if base:
                for root, _, files in os.walk(repo_dir):
                    if base in files:
                        return os.path.join(root, base)
            return target

        raw_path_val = arguments.get("path") or arguments.get("file_path") or arguments.get("filepath") or arguments.get("file") or ""
        if tool_name == "read_file":
            p = sanitize_path(raw_path_val)
            if not p or not os.path.exists(p):
                return f"Error: File not found: {raw_path_val}"
            with open(p, "r", encoding="utf-8", errors="ignore") as f:
                lines = f.readlines()
            s = max(1, int(arguments.get("start_line", 1))) - 1
            e = min(len(lines), int(arguments.get("end_line", min(len(lines), s + 80))))
            out = [f"{i+1}: {lines[i]}" for i in range(s, e)]
            return "".join(out)[:3500]
            
        elif tool_name == "edit_file":
            p = sanitize_path(raw_path_val)
            if not p or not os.path.exists(p):
                return f"Error: File not found: {raw_path_val}"
            old_str = str(arguments.get("old_string") or arguments.get("old_str") or "")
            new_str = str(arguments.get("new_string") or arguments.get("new_str") or "")
            if not old_str:
                return "Error: `old_string` is required and cannot be empty."
            with open(p, "r", encoding="utf-8", errors="ignore") as f:
                content = f.read()
            if old_str not in content:
                return f"Error: `old_string` not found in {raw_path_val}. Make sure you call read_file({raw_path_val}, start_line, end_line) first and copy exact indentation."
            content = content.replace(old_str, new_str, 1)
            with open(p, "w", encoding="utf-8") as f:
                f.write(content)
            return f"Successfully edited {raw_path_val}."
            
        elif tool_name == "run_command":
            cmd = str(arguments.get("command") or arguments.get("cmd") or "")
            if not cmd:
                return "Error: command is required."
            res = subprocess.run(cmd, shell=True, cwd=repo_dir, capture_output=True, text=True, timeout=30)
            out = (res.stdout + "\n" + res.stderr).strip()
            return out[:2500] if out else "(No output, exit code: " + str(res.returncode) + ")"
            
        elif tool_name == "get_status":
            res = subprocess.run(["git", "status", "--short"], cwd=repo_dir, capture_output=True, text=True)
            return res.stdout.strip() if res.stdout.strip() else "No changes."
            
        elif tool_name == "submit_patch":
            return "SUBMIT_PATCH_CALLED"
            
        else:
            return f"Error: Unknown tool '{tool_name}'."
    except Exception as e:
        return f"Execution Error in {tool_name}: {e}"

def parse_tool_call(response_text):
    """モデルの多様な出力形式 (JSON, Python AST, CLI構文, 代入ブロック) を万能パースする（複数ブロック時は末尾優先）"""
    # 1. <tool_call> または ```json { ... } (複数ある場合は末尾を採用)
    m_list = list(re.finditer(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', response_text, re.DOTALL))
    if not m_list:
        m_list = list(re.finditer(r'```(?:json)?\s*(\{.*?\})\s*```', response_text, re.DOTALL))
    for m in reversed(m_list):
        try:
            data = json.loads(m.group(1).strip())
            args_val = data.get("arguments", {})
            if isinstance(args_val, str):
                try: args_val = json.loads(args_val)
                except Exception: args_val = {"raw_input": args_val}
            elif not isinstance(args_val, dict):
                args_val = {"raw_input": str(args_val)}
            if "file_path" in args_val and "path" not in args_val:
                args_val["path"] = args_val.pop("file_path")
            return {"name": data.get("name", ""), "arguments": args_val}, {"has_tool": True, "parse_ok": True}
        except Exception:
            pass

    # コードブロック群の抽出（複数ある場合は末尾のブロックから優先評価し、復唱による過去コマンド誤実行を防止）
    code_blocks = re.findall(r'```(?:python|bash|sh)?\s*(.*?)\s*```', response_text, re.DOTALL)
    candidates = [b.strip() for b in reversed(code_blocks) if b.strip()] if code_blocks else [response_text.strip()]

    for code in candidates:
        # 2. Python AST パース (関数呼び出し構文: edit_file(...), read_file(...), etc.)
        try:
            tree = ast.parse(code)
            matched_calls = []
            for node in ast.walk(tree):
                if isinstance(node, ast.Call):
                    fn_name = node.func.id if isinstance(node.func, ast.Name) else ""
                    if fn_name in ["edit_file", "read_file", "run_command", "get_status", "submit_patch"]:
                        args = {}
                        for kw in node.keywords:
                            if isinstance(kw.value, ast.Constant):
                                args[kw.arg] = str(kw.value.value)
                        pos_names = {
                            'edit_file': ['path', 'old_string', 'new_string'],
                            'read_file': ['path', 'start_line', 'end_line'],
                            'run_command': ['command']
                        }
                        for i, a in enumerate(node.args):
                            if i < len(pos_names.get(fn_name, [])) and isinstance(a, ast.Constant):
                                args[pos_names[fn_name][i]] = str(a.value)
                        if "file_path" in args and "path" not in args:
                            args["path"] = args.pop("file_path")
                        if "filepath" in args and "path" not in args:
                            args["path"] = args.pop("filepath")
                        if fn_name == "edit_file":
                            missing_old = "old_string" not in args
                            if args.get("path") or args.get("old_string"):
                                matched_calls.append(({"name": fn_name, "arguments": args}, {"has_tool": True, "parse_ok": True, "missing_old_string": missing_old}))
                        else:
                            matched_calls.append(({"name": fn_name, "arguments": args}, {"has_tool": True, "parse_ok": True}))
            if matched_calls:
                return matched_calls[-1]
        except Exception:
            pass

        # 3. CLI 形式 read_file: read_file /workspace/path [start] [end]
        for line in reversed(code.splitlines()):
            line_s = line.strip()
            if line_s.startswith("read_file") and not line_s.startswith("read_file("):
                parts = line_s.split()
                if len(parts) >= 2:
                    p_clean = parts[1].replace("'", "").replace('"', '')
                    args = {"path": p_clean}
                    if len(parts) >= 3 and parts[2].isdigit(): args["start_line"] = int(parts[2])
                    if len(parts) >= 4 and parts[3].isdigit(): args["end_line"] = int(parts[3])
                    return {"name": "read_file", "arguments": args}, {"has_tool": True, "parse_ok": True}

        # 4. CLI 形式 edit_file または代入ブロック (ASTフォールバック)
        if "edit_file" in code:
            path_arg = ""
            for line in code.splitlines():
                line_s = line.strip()
                if line_s.startswith("edit_file") and not line_s.startswith("edit_file("):
                    parts = line_s.split()
                    if len(parts) >= 2:
                        path_arg = parts[1].replace("'", "").replace('"', '')
                        break
            if not path_arg:
                m_path = re.search(r'(?:file_)?path\s*=\s*["\']([^"\']+)["\']', code)
                if m_path: path_arg = m_path.group(1)
            m_old = re.search(r'old_string\s*=\s*(?:"""|\'\'\'|"|\')([\s\S]*?)(?:"""|\'\'\'|"|\')', code)
            m_new = re.search(r'new_string\s*=\s*(?:"""|\'\'\'|"|\')([\s\S]*?)(?:"""|\'\'\'|"|\')', code)
            old_val = m_old.group(1) if m_old else ""
            new_val = m_new.group(1) if m_new else ""
            if not old_val and "old_string" in code and "new_string" in code:
                try:
                    s_old = code[code.index("old_string"):]
                    old_val = s_old[:s_old.index("new_string")].split("=", 1)[1].strip().replace(chr(39), "").replace(chr(34), "")
                    s_new = code[code.index("new_string"):]
                    new_val = s_new.split("=", 1)[1].strip().replace(chr(39), "").replace(chr(34), "")
                except Exception:
                    pass
            if path_arg or old_val:
                return {"name": "edit_file", "arguments": {"path": path_arg, "old_string": old_val, "new_string": new_val}}, {"has_tool": True, "parse_ok": True}

        # 5. シェルコマンド形式 (pytest, git, python, etc.)
        first_line = code.splitlines()[0].strip() if code.splitlines() else ""
        if any(first_line.startswith(cmd) for cmd in ["pytest", "git", "python", "python3", "cat", "grep"]):
            return {"name": "run_command", "arguments": {"command": code}}, {"has_tool": True, "parse_ok": True}
        if "submit_patch" in first_line:
            return {"name": "submit_patch", "arguments": {}}, {"has_tool": True, "parse_ok": True}

    return None, {"has_tool": False, "error": "No recognized tool format"}

print("[Engine] ✅ 万能マルチ形式パーサー & 実環境ツールエンジン準備完了。")


In [ ]:
# 5. 【検証 1】波括弧 `{` の Google ADK / テンプレートエンジン破壊直接テスト
print("="*75)
print("🔬 検証 1: プロンプト内の波括弧 { ... } がテンプレートエンジンに与える影響")
print("="*75)

prompt_with_curly = '''System instruction: Use tool format:
<tool_call>
{"name": "edit_file", "arguments": {"path": "rich/console.py", "old_string": "x = 1", "new_string": "x = 2"}}
</tool_call>
'''

# 1. Python str.format() テスト (ADK 内部で使われる場合)
try:
    res = prompt_with_curly.format(issue="dummy issue")
    print("  [str.format] ✅ 正常レンダリング")
except Exception as e:
    print(f"  [str.format] ❌ 致命的例外検知: {type(e).__name__} ({e})")
    print("    ➔ 根拠実証: プロンプト内に波括弧があると、str.format を使うハーネスは KeyError でクラッシュします！")

# 2. string.Template テスト
from string import Template
try:
    t = Template(prompt_with_curly)
    res2 = t.safe_substitute(issue="dummy issue")
    print("  [string.Template] ✅ 正常 (safe_substitute)")
except Exception as e:
    print(f"  [string.Template] ❌ 例外: {e}")

print("="*75)


In [ ]:
# 6. 【本番仕様 A/B 最終検証】共通ツール定義 + 2 大本番プロンプト (V9.2 Full vs V15 Hybrid Production)

# 本番 ADK ハーネスが自動注入するツールシグネチャ定義（実験の交絡因子を排除するため全バリアントに共通付与）
COMMON_TOOL_HEADER = """
# AVAILABLE TOOLS & EXACT CALL SIGNATURES
Emit ONE tool call per turn inside a ```python code block using these exact keyword argument names:
- run_command(command="...")
- read_file(path="...", start_line=1, end_line=60)
- edit_file(path="...", old_string="...", new_string="...")
- get_status()
- submit_patch()
"""

# --- [Variant 1: V9.2_Production_Baseline (現行 submission.zip フルプロンプト / LB 0.17 実績)] ---
PROMPT_V9_2_PROD = """You are an autonomous software engineer. Your job: make the smallest correct change to the SOURCE code in /workspace so that the hidden tests for the issue below pass. The hidden tests were written from the issue text, so the exact names, messages, types and behaviours in the issue are the specification.

# HARD LIMITS
- 28 tool calls and 8 minutes in total. When either runs out, the working tree is graded as it is. An edit that exists counts, even if you never call `submit_patch`. An unedited tree always scores zero.
- EARLY-EDIT TRIGGER (DEADLINE AT CALL 8): You MUST execute `edit_file` on or before Call 8. Never keep reading files or thinking past Call 8 without applying an edit. An imperfect or partial edit can be graded and score partial credit; an empty patch (zero edits) scores ZERO.
- Never create, modify or delete test files (`test_*.py`, `*_test.py`, anything under `tests/`, `pytest.ini`, `conftest.py`). This is an automatic failure.
- Scratch files go in /tmp only. Any file left under /workspace becomes part of the graded patch.
- Never run a full test suite (bare `pytest`, `pytest .`, `unittest discover`). Always name one test file, and add `-k` when you can.
- Never look outside /workspace. All dependencies are already installed.
- Do NOT restate, echo, or summarize code or files you just read. Read silently, then act immediately.

# CALL PLAN
- Calls 1-7: locate and read target regions (max 40-80 lines per read).
- Call 8: HARD TRIGGER - MANDATORY FIRST EDIT. Apply your best-guess minimal patch to the primary target file now. Do not hesitate.
- Calls 9-15: verify and correct (including coupled files and parallel sync/async implementations).
- Calls 16-20: final check, clean, and call `submit_patch`.

# PROCEDURE
1. Extract the specification: exact file, function, class, error message, edge cases.
2. Locate: Use `git grep -n "TOKEN" -- '*.py' | head -20`. Read ~40 lines around target with `read_file`.
3. Edit & Parallel Implementations:
   - Unique Multi-Line Block Match: Include 3 to 5 lines of surrounding context in `old_string`.
   - Parallel Implementations (Sync / Async): Apply change to both sync and async variants if present.
   - Coupled Multi-File Edits: Update immediate consumers if changing signatures (<= 3 files).
4. Verify: Run `python -m pytest tests/path/test_x.py -x -q -k keyword`.
5. Final check and submit: Run `git status --short && git diff`, then `submit_patch`.
""" + COMMON_TOOL_HEADER

# --- [Variant 2: V15_Hybrid_Production (本番提出候補: フル仕様 + Pinpoint Read-Before-Edit + 自己修復 + 即時遷移)] ---
PROMPT_V15_HYBRID_BODY = """You are an autonomous software engineer. Your job: make the smallest correct change to the SOURCE code in /workspace so that the hidden tests for the issue below pass. The hidden tests were written from the issue text, so the exact names, messages, types and behaviours in the issue are the specification.

# HARD LIMITS
- 28 tool calls and 8 minutes in total. When either runs out, the working tree is graded as it is. An edit that exists counts, even if you never call `submit_patch`. An unedited tree always scores zero.
- Never create, modify or delete test files (`test_*.py`, `*_test.py`, anything under `tests/`, `pytest.ini`, `conftest.py`). This is an automatic failure.
- Scratch files go in /tmp only. Any file left under /workspace becomes part of the graded patch.
- Never run a full test suite (bare `pytest`, `pytest .`, `unittest discover`). Always name one test file, and add `-k` when you can.
- Never look outside /workspace. All dependencies are already installed.
- Do NOT restate, echo, or repeat previous tool calls in your response. Output only the single next tool call.

# CORE PROTOCOL (PINPOINT READ-BEFORE-EDIT & ANTI-LOOP)
1. NO BLIND EDITS: Never call `edit_file` on a file region you have not read yet. First locate exact line numbers with `run_command(command="git grep -n 'TOKEN' -- '*.py' | head -20")`.
2. IMMEDIATE READ TRANSITION (NO REPEATED GREP): As soon as `git grep` shows a candidate file and line number (e.g., line N), DO NOT run another grep. Immediately call `read_file(path="...", start_line=max(1, N-20), end_line=N+40)` in the very next turn.
3. EXACT INDENTATION COPY: In `edit_file(path="...", old_string="...", new_string="...")`, copy 3 to 5 lines directly from the `read_file` output into `old_string` (without line numbers) so indentation matches character-by-character.
4. SELF-CORRECTION ON MISMATCH: If `edit_file` returns `old_string not found`, NEVER repeat the same `edit_file` call. Immediately call `read_file(path="...", start_line=..., end_line=...)` on that exact line range to inspect the real indentation, then apply the corrected `edit_file`.
5. STOP ON PASS: Run a single focused test (`python -m pytest tests/path/test_x.py -x -q -k keyword`). As soon as the focused test passes, stop investigating and call `submit_patch()` immediately.

# MULTI-FILE & PARALLEL RULES
- Parallel Implementations (Sync / Async): If the behaviour exists in both sync and async variants (`Client`/`AsyncClient`, `def`/`async def`), apply the fix to both.
- Coupled Multi-File Edits: If you change a signature or dataclass, update its immediate consumer (<= 3 files).
- Resilient Edit Recovery: If `edit_file` fails twice after reading, you may apply the exact replacement via `run_command` using `python3 -c`.

# STEP-BY-STEP WORKFLOW
Step 1 (Locate): `run_command(command="git grep -n 'SYMBOL' -- '*.py' | head -20")`
Step 2 (Pinpoint Read): `read_file(path="target.py", start_line=..., end_line=...)`
Step 3 (Exact Edit): `edit_file(path="target.py", old_string="...", new_string="...")`
Step 4 (Focused Verify): `run_command(command="python -m pytest tests/test_target.py -x -q -k keyword")`
Step 5 (Submit): `submit_patch()`
"""
PROMPT_V15_HYBRID_PROD = PROMPT_V15_HYBRID_BODY + COMMON_TOOL_HEADER

VARIANTS = {
    "V9.2_Production_Baseline (LB 0.17)": PROMPT_V9_2_PROD,
    "V15_Hybrid_Production (Candidate)": PROMPT_V15_HYBRID_PROD
}
print("[Prompt] ✅ 本番仕様 2 大バリアント定義完了:", list(VARIANTS.keys()))


In [ ]:
# 7. クリーンな Rich 実タスクデータセットの選定
import urllib.request

LOCBENCH_FILE = "/content/gemma4-locbench-v1.json"
LOCBENCH_URL = "https://raw.githubusercontent.com/chiba244qq/gemma4-agent-dashboard/main/data/locbench/gemma4-locbench-v1.json"

if not os.path.exists(LOCBENCH_FILE):
    print("[LocBench] GitHub から LocBench-v1 をダウンロード中...")
    urllib.request.urlretrieve(LOCBENCH_URL, LOCBENCH_FILE)

with open(LOCBENCH_FILE, "r", encoding="utf-8") as f:
    locbench_data = json.load(f)

all_records = locbench_data.get("records", [])
# コミュニティ監査で健全性が証明された Textualize/rich タスクのみを抽出
rich_records = [r for r in all_records if r.get("repo") == "Textualize/rich"]
print(f"[LocBench] ✅ 読み込み完了: 全 {len(all_records)} 件中、クリーンな Rich タスク: {len(rich_records)} 件")

# 代表実タスク 3 件を選定 (Smoke A/B テスト)
# rich_4077: proxy isatty (test_file_proxy.py)
# rich_4079: inline table code (test_markdown.py)
# rich_4076: preserve newlines (test_ansi.py)
SELECTED_TASKS = rich_records[1:4]
print(f"[LocBench] 🎯 選定された評価タスク:")
for t in SELECTED_TASKS:
    print(f"  - {t['instance_id']}: {t.get('verification_test_files')} ({t.get('problem_statement_excerpt')[:60]}...)")


In [ ]:
# 8. 【マルチターン E2E 自律実行 ＆ 実際の pytest 判定ループ (max_turns=10)】

def evaluate_task_variant(task, variant_name, system_prompt, max_turns=10):
    print(f"\n{'='*65}")
    print(f"▶ Task: {task['instance_id']} | Variant: {variant_name}")
    print(f"{'='*65}")
    
    # リポジトリを完全クリーンアップ & 当該コミットにチェックアウト
    subprocess.run(["git", "checkout", "-f", task["base_commit"]], cwd=REPO_DIR, capture_output=True)
    subprocess.run(["git", "clean", "-fd"], cwd=REPO_DIR, capture_output=True)
    
    issue_text = f"Issue Description for {task['instance_id']}:\n" + task.get("problem_statement_excerpt", "Fix the bug in the codebase.")
    if task.get("verification_test_files"):
        issue_text += f"\nRelated test files to verify: {', '.join(task['verification_test_files'])}"
        
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": issue_text}
    ]
    
    total_latency = 0.0
    parse_errors = 0
    missing_old_str_count = 0
    patch_applied = False
    prev_call_sig = None
    
    for turn in range(1, max_turns + 1):
        # 最終ターンでまだパッチ未適用のときのみリマインダーを送出
        if turn == max_turns and not patch_applied:
            messages.append({"role": "user", "content": f"[SYSTEM REMINDER: Final turn {turn}/{max_turns} reached without an applied patch. You MUST call edit_file now using the exact lines from read_file.]"})
            
        resp, lat = call_llm(messages, temperature=0.2, max_new_tokens=1024)
        total_latency += lat
        print(f"  [Turn {turn}/{max_turns}] 推論所要: {lat:.2f}s | 出力長: {len(resp)} 文字")
        
        tool_call, audit = parse_tool_call(resp)
        if audit.get("missing_old_string"):
            missing_old_str_count += 1
        if not audit.get("parse_ok", True):
            parse_errors += 1
            
        if not tool_call:
            print(f"    ⚠️ ツール呼出なし: {resp[:100]}...")
            messages.append({"role": "assistant", "content": resp if resp else "(thinking)"})
            messages.append({"role": "user", "content": 'Please emit a single valid tool call inside a ```python block.'})
            continue
            
        t_name = tool_call.get("name", "")
        t_args = tool_call.get("arguments", {})
        if not isinstance(t_args, dict):
            t_args = {"raw_input": str(t_args)}
            
        # 透明性の高い詳細ログ表示
        if t_name == "run_command":
            arg_summary = f"cmd={repr(str(t_args.get('command', ''))[:60])}"
        elif t_name == "read_file":
            arg_summary = f"path={repr(t_args.get('path'))}, lines={t_args.get('start_line', 1)}..{t_args.get('end_line', 'end')}"
        elif t_name == "edit_file":
            old_preview = str(t_args.get('old_string', '')).strip().splitlines()[0][:35] if t_args.get('old_string') else ''
            arg_summary = f"path={repr(t_args.get('path'))}, old={repr(old_preview)}"
        else:
            arg_summary = str(list(t_args.keys()))
        print(f"    🛠️ Tool Call: {t_name}({arg_summary})")
        
        if t_name == "submit_patch":
            print("    🏁 submit_patch が呼び出されました。探索を終了します。")
            break
            
        tool_out = execute_tool(t_name, t_args, REPO_DIR)
        print(f"      ↳ Result: {tool_out[:100].strip()}...")
        if t_name == "edit_file" and "Successfully edited" in tool_out:
            patch_applied = True
            
        # 同一コマンド反復検知ガード (Anti-Loop Feedback)
        cur_call_sig = f"{t_name}:{json.dumps(t_args, sort_keys=True)}"
        if cur_call_sig == prev_call_sig:
            tool_out += "\n\n[SYSTEM WARNING: You just executed this exact same tool call in the previous turn. Do NOT repeat it. If you located a file/line, call read_file(path, start_line, end_line) immediately. If you already read the lines, call edit_file(path, old_string, new_string) with the exact indentation.]"
        prev_call_sig = cur_call_sig
            
        messages.append({"role": "assistant", "content": resp})
        messages.append({"role": "user", "content": f"Tool output:\n{tool_out}"})
    
    # 実際の git diff を取得
    diff_res = subprocess.run(["git", "diff"], cwd=REPO_DIR, capture_output=True, text=True)
    diff_text = diff_res.stdout.strip()
    has_diff = len(diff_text) > 0
    
    # 実際の pytest を実行して合否を判定
    test_files = task.get("verification_test_files", [])
    pytest_pass = False
    pytest_log = ""
    
    if test_files and os.path.exists(os.path.join(REPO_DIR, test_files[0])):
        cmd = ["pytest", "-q", "--maxfail=1"] + test_files
        py_res = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True, timeout=60)
        pytest_raw_pass = (py_res.returncode == 0)
        pytest_log = (py_res.stdout + "\n" + py_res.stderr)[-500:].strip()
    else:
        pytest_raw_pass = False
        pytest_log = "No test files found."
        
    if not has_diff:
        pytest_pass = False
        verdict = "⚠️ NO_PATCH (Score 0)"
    else:
        pytest_pass = pytest_raw_pass
        verdict = "✅ PASS" if pytest_pass else "❌ FAIL"
        
    print(f"  📊 Task 結果: パッチ生成={has_diff} | pytest判定={verdict}")
        
    
    return {
        "task_id": task["instance_id"],
        "variant": variant_name,
        "pytest_pass": pytest_pass,
        "has_diff": has_diff,
        "turns_used": turn,
        "total_latency_sec": round(total_latency, 2),
        "avg_step_latency_sec": round(total_latency / max(1, turn), 2),
        "parse_errors": parse_errors,
        "missing_old_str_count": missing_old_str_count,
        "pytest_log": pytest_log
    }

all_eval_results = []
for t in SELECTED_TASKS:
    for v_name, v_prompt in VARIANTS.items():
        res = evaluate_task_variant(t, v_name, v_prompt, max_turns=10)
        all_eval_results.append(res)
print("\n🎉 全 A/B 実機検証ランが完了しました！")


In [ ]:
# 9. 結果集計・統計比較表の出力 & GitHub 自動同期
import pandas as pd
from tabulate import tabulate

df = pd.DataFrame(all_eval_results)
summary = []
for v_name in VARIANTS.keys():
    v_df = df[df["variant"] == v_name]
    n = len(v_df)
    if n == 0:
        continue
    pass_rate = (v_df["pytest_pass"].sum() / n) * 100
    patch_rate = (v_df["has_diff"].sum() / n) * 100
    avg_latency = v_df["avg_step_latency_sec"].mean()
    avg_turns = v_df["turns_used"].mean()
    total_parse_err = v_df["parse_errors"].sum()
    total_missing_old = v_df["missing_old_str_count"].sum()
    
    summary.append({
        "Variant": v_name,
        "Tasks": n,
        "Pass Rate (%)": f"{pass_rate:.1f}%",
        "Patch Rate (%)": f"{patch_rate:.1f}%",
        "Parse Errors": int(total_parse_err),
        "Missing old_string": int(total_missing_old),
        "Avg Step Latency": f"{avg_latency:.2f}s",
        "Avg Turns": f"{avg_turns:.1f}"
    })

sum_df = pd.DataFrame(summary)
print("="*85)
print("🏆 【クリーン実タスク (Rich) A/B 検証結果サマリー】")
print("="*85)
print(tabulate(sum_df, headers="keys", tablefmt="github", showindex=False))

# GitHub への自動プッシュ
now_str = time.strftime("%Y-%m-%d %H:%M:%S JST")
output_data = {
    "timestamp": now_str,
    "model": MODEL_ID,
    "eval_mode": "clean_rich_tasks",
    "summary": summary,
    "detailed_results": all_eval_results
}

try:
    github_pat = userdata.get('GITHUB_PAT')
    if github_pat:
        import requests, base64
        repo = "chiba244qq/gemma4-agent-dashboard"
        path = "data/locbench_ab_test_results.json"
        api_url = f"https://api.github.com/repos/{repo}/contents/{path}"
        headers = {"Authorization": f"token {github_pat}", "Accept": "application/vnd.github.v3+json"}
        sha = None
        get_res = requests.get(api_url, headers=headers)
        if get_res.status_code == 200:
            sha = get_res.json().get("sha")
        content_bytes = json.dumps(output_data, indent=2, ensure_ascii=False).encode('utf-8')
        payload = {
            "message": f"[LocBench A/B Sync] Rich クリーンタスク評価結果更新 ({now_str})",
            "content": base64.b64encode(content_bytes).decode('utf-8')
        }
        if sha:
            payload["sha"] = sha
        put_res = requests.put(api_url, headers=headers, json=payload)
        if put_res.status_code in [200, 201]:
            print(f"\n[Sync] 🎉 GitHub ({path}) への自動プッシュに成功しました！")
        else:
            print(f"\n[Sync Warning] GitHub プッシュ失敗: {put_res.status_code}")
except Exception as e:
    print(f"\n[Sync Notice] GitHub 同期スキップ: {e}")
